# Kaggle caption: OmniShotCut → SpeedPredictor → InternVL3.5 + vLLM

Bật Internet và GPU. Cell 2 cài runtime vLLM riêng; có thể thay Torch có sẵn bằng bộ tương thích. Notebook không import Torch/vLLM vào kernel. Cell 3 xác nhận GPU thực tế qua process mới (tôn trọng CUDA_VISIBLE_DEVICES), in tên/VRAM và kế hoạch inference. Một GPU: TP=1, tối đa 3 shot đồng thời, 6 shot/batch. Hai GPU trở lên: dùng hai GPU đầu, TP=2 chia model trên cả hai, tối đa 6 shot đồng thời, 12 shot/batch. Mặc định token budget 4096/8192 cho 1/2 GPU. Đây là cấu hình thử nghiệm, chưa benchmark Kaggle; TP không bảo đảm tăng tốc gấp đôi.

Mặc định chỉ hiện progress bar cho model loading, OMNI decode/windows, SPEED theo shot và CAPTION theo shot. Các bar được dựng trong notebook qua giao thức progress stdout để không tạo hàng nghìn dòng. OMNI decode là bar 0/1 video vì upstream chưa cung cấp tiến độ decode từng frame. Log INFO từng frame/pass/score bị ẩn; cảnh báo/lỗi/traceback vẫn hiện. Cell 3: PRINT_CAPTIONS=True để in caption cuối từng shot; VERBOSE_LOGS=True để khôi phục log chi tiết phục vụ debug. Không lưu file log/score/label.

Giữ prompt của người dùng, 1.5 FPS, tối đa 16 frame, một tile 448×448/frame. Lọc RGB histogram 8×8×8 trên frame gốc trước resize, loại khi <5 bin có màu, đúng 5 bin vẫn giữ; không lấy bù. Shot bị lọc hết không gọi generate và không xuất segment. Nếu toàn video bị lọc, JSON là [] hợp lệ. Prefetch chỉ giữ một batch chờ; engine giữ xuyên suốt lô. Checkpoint mỗi batch thành công; thay batch/VRAM/GPU/log không đổi run ID để tiếp tục được tiến độ cũ. Chỉ JSON caption được upload, file shot/JSON giữ local và video tạm được dọn sau xử lý.

OmniShotCut được giải phóng trước SpeedPredictor; speed dùng GPU 0 trong process riêng, sau đó giải phóng trước vLLM. Predictor giữ 16 frame, 7.5 FPS tham chiếu, 3 lượt thích nghi và FPS nguồn thực. Shot <2.133s là unknown. Ngưỡng >1.8 timelapse, <0.6 slow-motion. Nhãn chỉ chèn vào prefix sau caption; JSON cuối vẫn chỉ có video_id/start/end/caption.

Chạy lại cell 2 → 3 → 4 → 5 khi cập nhật. Để so sánh cùng video, chọn folder Drive đích riêng vì mặc định ID đã có JSON trên Drive sẽ bị bỏ qua. OOM: giảm MAX_NUM_SEQS/MAX_NUM_BATCHED_TOKENS hoặc bật ENFORCE_EAGER rồi chạy lại cell 3 + 5. Không tự giảm frame và không retry engine đã crash. Compile/load có thể lâu ở lần đầu; thanh model loading hiện thời gian chờ, không phải tiến độ compile từng kernel.


In [ ]:
# 1. Cấu hình
import importlib.metadata, importlib.util, json, os, re, shutil, subprocess, sys, urllib.request, zipfile
from pathlib import Path
REPO_URL = 'https://github.com/iamtro-2006/event-retrieval-system.git'
REPO_BRANCH = 'vbs-dev'  # nhánh chứa pipeline caption
DATASET = 'V3C'
START_VIDEO_ID = 1
END_VIDEO_ID = 10
VIDEO_EXTENSION = '.mp4'
SOURCE_ROOT_FOLDER_ID = '19b7skL2b0PRJWWweWFh8qVNTLQDF6x9K'
DEST_ROOT_FOLDER_ID = '1I97-3-Y_V0-dfk-tPS4g7H8lpAHyeu2r'
RCLONE_CONFIG_SOURCE = Path('/kaggle/input/datasets/vnnoname/rclone-new/rclone.conf')
RCLONE_REMOTE = 'iamtro'
UPLOAD_BATCH_SIZE = 10
OVERWRITE_REMOTE = False

def video_id(number):
    return f'{number:05d}'

if DATASET != 'V3C' or not 1 <= START_VIDEO_ID <= END_VIDEO_ID <= 99999:
    raise ValueError('Cần 1 <= START_VIDEO_ID <= END_VIDEO_ID <= 99999 cho V3C')
if SOURCE_ROOT_FOLDER_ID == DEST_ROOT_FOLDER_ID:
    raise ValueError('Folder Drive nguồn và đích phải khác nhau')
if not RCLONE_CONFIG_SOURCE.is_file():
    raise FileNotFoundError(f'Chưa gắn rclone config: {RCLONE_CONFIG_SOURCE}')
WORK = Path('/kaggle/working/caption_pipeline')
WORK.mkdir(parents=True, exist_ok=True)
JOBS_ROOT = WORK / 'jobs'
JOBS_ROOT.mkdir(exist_ok=True)
RCLONE_CONFIG = WORK / 'rclone.conf'
shutil.copyfile(RCLONE_CONFIG_SOURCE, RCLONE_CONFIG)
RCLONE_CONFIG.chmod(0o600)
REPO_DIR = WORK / 'event-retrieval-system'
BACKEND_DIR = REPO_DIR / 'backend'
print('Workspace:', WORK)


In [ ]:
# 2. Clone/pull repo, cài dependencies chung + OmniShotCut và chuẩn bị rclone
def run(args, *, cwd=None, env=None):
    from collections import deque
    import queue
    import signal
    import threading
    import time
    args = [str(item) for item in args]
    print('+', ' '.join(args[:5]), '...' if len(args) > 5 else '', flush=True)
    tail = deque(maxlen=80)
    progress_bars = {}
    progress_prefix = '__CAPTION_PROGRESS__='
    quiet_pipeline = bool(env and env.get('CAPTION_PROGRESS_PROTOCOL') == '1'
                          and env.get('CAPTION_VERBOSE', '0') != '1')
    if quiet_pipeline:
        from tqdm.auto import tqdm
    messages = queue.Queue()
    started = time.monotonic()
    last_output = started
    with subprocess.Popen(args, cwd=cwd, env=env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, errors='replace',
                          bufsize=1) as process:
        def pump():
            try:
                for line in process.stdout:
                    messages.put(line)
            finally:
                messages.put(None)
        threading.Thread(target=pump, daemon=True).start()
        eof = False
        try:
            while not eof or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                except queue.Empty:
                    for bar in progress_bars.values():
                        if not bar.disable:
                            bar.refresh()
                    if time.monotonic() - last_output >= 30 and not any(not b.disable for b in progress_bars.values()):
                        status = process.poll()
                        print(f'[RUN-HEARTBEAT] pid={process.pid} '
                              f'elapsed={time.monotonic()-started:.0f}s '
                              f'status={"running" if status is None else status}; '
                              'no new child output for 30s (not a progress estimate)', flush=True)
                        last_output = time.monotonic()
                    continue
                if line is None:
                    eof = True
                    continue
                if quiet_pipeline and line.startswith(progress_prefix):
                    event = json.loads(line[len(progress_prefix):])
                    key = event['key']
                    if key not in progress_bars:
                        progress_bars[key] = tqdm(total=event['total'], initial=event['n'],
                            desc=f"{event['stage']} {event['video']}".strip(), unit=event['unit'],
                            dynamic_ncols=True, mininterval=.5, leave=True)
                    bar = progress_bars[key]
                    bar.set_postfix_str(event.get('postfix', ''), refresh=False)
                    bar.update(max(0, event['n'] - bar.n))
                    if event['status'] in ('done', 'error'):
                        if event['status'] == 'error':
                            bar.set_description(bar.desc + ' FAILED')
                        bar.close()
                else:
                    # Keep warnings/tracebacks; suppress only recognizable INFO/DEBUG chatter.
                    noise = quiet_pipeline and re.search(
                        r'^(?:\d{4}-\d\d-\d\d\s+\S+\s+(?:INFO|DEBUG)\b|\[(?:INFO|DEBUG)\s*\]|(?:INFO|DEBUG)\s)', line)
                    noise = noise or (quiet_pipeline and re.search(
                        r'(?:Loading (?:safetensors|checkpoint)|Capturing CUDA graph|Processed prompts).*\|', line))
                    if not noise:
                        if quiet_pipeline:
                            tqdm.write(line.rstrip())
                        else:
                            print(line, end='', flush=True)
                    tail.append(line)
                last_output = time.monotonic()
            exit_code = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            for bar in progress_bars.values():
                if not bar.disable:
                    bar.close()
            print('[RUN-END] interrupted by user', flush=True)
            raise
    for bar in progress_bars.values():
        if not bar.disable:
            bar.close()
    detail = ''
    if exit_code < 0:
        try:
            detail = f' signal={signal.Signals(-exit_code).name}'
        except ValueError:
            detail = f' signal={-exit_code}'
    print(f'[RUN-END] pid={process.pid} exit_code={exit_code}{detail} '
          f'elapsed={time.monotonic()-started:.1f}s', flush=True)
    if exit_code:
        raise RuntimeError(f'Command exited with code {exit_code}{detail}: {args[:5]}\n'
                           + ''.join(tail))
    return exit_code

if not REPO_DIR.exists():
    run(['git', 'clone', '--depth', '1', '--single-branch', '--branch', REPO_BRANCH, REPO_URL, REPO_DIR])
else:
    branch = subprocess.check_output(['git', '-C', str(REPO_DIR), 'branch', '--show-current'], text=True).strip()
    if branch != REPO_BRANCH:
        raise RuntimeError(f'Checkout đang ở {branch}, cần {REPO_BRANCH}')
    run(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
if not (BACKEND_DIR / 'src/caption_extraction/pipeline/extract_caption.py').is_file():
    raise RuntimeError('Nhánh GitHub chưa có caption extraction; hãy push code lên vbs-dev')
print('Caption commit:', commit)

# Bộ dependencies vLLM riêng; không import Torch trong kernel notebook.
def installed_version(package):
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return None

requirements = BACKEND_DIR / 'requirements-caption.txt'
if (installed_version('vllm') != '0.11.2' or installed_version('transformers') != '4.57.1'
        or (installed_version('torch') or '').split('+')[0] != '2.9.0'
        or any(installed_version(name) is None for name in
               ('decord', 'PyNvVideoCodec', 'timm', 'accelerate', 'PyYAML', 'ffmpeg-python', 'einops', 'sentencepiece'))):
    run([sys.executable, '-m', 'pip', 'install', '--upgrade-strategy', 'only-if-needed', '-r', requirements])
if importlib.util.find_spec('omnishotcut') is None:
    run([sys.executable, '-m', 'pip', 'install', '--upgrade-strategy', 'only-if-needed',
         'git+https://github.com/UVA-Computer-Vision-Lab/OmniShotCut.git'])
import yaml
run([sys.executable, '-u', '-c',
     'import torch, transformers, vllm; '
     'assert torch.cuda.is_available(), "Enable GPU in Kaggle Settings"; '
     'print("Visible GPUs:", [(i, torch.cuda.get_device_name(i)) for i in range(torch.cuda.device_count())], '
     '"Torch:", torch.__version__, "vLLM:", vllm.__version__, flush=True)'])

rclone = shutil.which('rclone')
if rclone is None and shutil.which('apt-get'):
    try:
        run(['apt-get', 'update', '-qq'])
        run(['apt-get', 'install', '-y', '-qq', 'rclone'])
        rclone = shutil.which('rclone')
    except subprocess.CalledProcessError:
        pass
if rclone is None:
    release = 'v1.75.2'
    archive = WORK / f'rclone-{release}-linux-amd64.zip'
    url = f'https://github.com/rclone/rclone/releases/download/{release}/{archive.name}'
    with urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'kaggle-caption'}), timeout=120) as src, archive.open('wb') as dst:
        shutil.copyfileobj(src, dst)
    with zipfile.ZipFile(archive) as zf:
        matches = [item for item in zf.infolist() if item.filename.endswith('/rclone') and not item.is_dir()]
        if len(matches) != 1:
            raise RuntimeError('Không tìm thấy binary rclone trong ZIP release')
        rclone = WORK / 'rclone'
        with zf.open(matches[0]) as src, rclone.open('wb') as dst:
            shutil.copyfileobj(src, dst)
    rclone.chmod(0o755)
run([rclone, 'version'])


In [ ]:
# 3. InternVL3.5 + vLLM: sửa tham số/prompt rồi chạy lại cell này + cell 5.
import hashlib
import math
import yaml

CAPTION_MODEL = 'OpenGVLab/InternVL3_5-2B'
FRAME_SAMPLE_FPS = 1.5
MAX_FRAMES = 16
MIN_COLOR_BINS = 5          # Loai frame co <5 bins mau; dung 5 bins van giu
CAPTION_BATCH_SIZE = 'auto' # 6 on one GPU, 12 on two GPUs
TENSOR_PARALLEL_SIZE = 'auto' # Shard model over 2 GPUs when available
MAX_NUM_SEQS = 'auto'        # 3 on one GPU, 6 across two GPUs
GPU_MEMORY_UTILIZATION = 0.88
MAX_MODEL_LEN = 6144
MAX_NUM_BATCHED_TOKENS = 'auto' # 4096 / 8192 for 1 / 2 GPUs
ENABLE_PREFIX_CACHING = True
ENFORCE_EAGER = False       # True bỏ CUDA graph/compile nếu cần giảm VRAM
MAX_NEW_TOKENS = 220
VERBOSE_LOGS = False         # True restores detailed per-frame/pass diagnostics
PRINT_CAPTIONS = False       # True prints each final caption in addition to progress bars
TIMELAPSE_THRESHOLD = 1.8    # score > ngưỡng này
SLOW_MOTION_THRESHOLD = 0.6  # score < ngưỡng này

CAPTION_PROMPT = (
    'Describe the following video segment in English. Begin with "The video segment". In 1-4 detailed sentences, focusing main visible subjects, their distinctive details and positions, the relevant surroundings at that moment. Describe what happens from the beginning to the end in reasonable chronological order, including meaningful changes, interactions between subjects, and camera movement when visible. Be specific about observable actions, scene, interaction and subject attribute details and identity rather than giving a broad summary. Include only what can be seen; omit uncertain or absent details without mentioning their absence. DO NOT GUESS any event, action or intent.'
)

vllm_config = {
    'gpu_memory_utilization': GPU_MEMORY_UTILIZATION,
    'max_model_len': MAX_MODEL_LEN,
    'tensor_parallel_size': TENSOR_PARALLEL_SIZE,
    'max_num_seqs': MAX_NUM_SEQS,
    'max_num_batched_tokens': MAX_NUM_BATCHED_TOKENS,
    'enable_prefix_caching': ENABLE_PREFIX_CACHING,
    'enforce_eager': ENFORCE_EAGER,
}
# Shared validation imports no Torch/vLLM into the notebook kernel.
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))
import importlib
import src.caption_extraction.pipeline.internvl as caption_runtime
caption_runtime = importlib.reload(caption_runtime)
validate_settings = caption_runtime.validate_settings
vllm_config = validate_settings(CAPTION_MODEL, MAX_FRAMES, CAPTION_BATCH_SIZE, vllm_config)
# Detect GPUs in a fresh process, respecting CUDA_VISIBLE_DEVICES.
gpu_probe = subprocess.check_output([sys.executable, '-c',
    'import json, torch; print(json.dumps([dict(index=i, name=torch.cuda.get_device_name(i), '
    'vram_gib=round(torch.cuda.get_device_properties(i).total_memory/2**30,2)) '
    'for i in range(torch.cuda.device_count())]))'], text=True)
VISIBLE_GPUS = json.loads(gpu_probe)
gpu_plan = caption_runtime.resolve_gpu_settings(len(VISIBLE_GPUS), vllm_config)
resolved_batch = 2 * gpu_plan['max_num_seqs'] if CAPTION_BATCH_SIZE == 'auto' else CAPTION_BATCH_SIZE
print('Confirmed GPUs:', VISIBLE_GPUS, flush=True)
print('Caption plan: TP=', gpu_plan['tensor_parallel_size'], '| concurrency=', gpu_plan['max_num_seqs'],
      '| batch=', resolved_batch, flush=True)
if type(VERBOSE_LOGS) is not bool or type(PRINT_CAPTIONS) is not bool:
    raise ValueError('VERBOSE_LOGS and PRINT_CAPTIONS must be boolean')
if type(MIN_COLOR_BINS) is not int or not 1 <= MIN_COLOR_BINS <= 512:
    raise ValueError('MIN_COLOR_BINS must be an integer in [1, 512]')
if not math.isfinite(FRAME_SAMPLE_FPS) or FRAME_SAMPLE_FPS <= 0:
    raise ValueError('FRAME_SAMPLE_FPS must be positive and finite')
if type(MAX_NEW_TOKENS) is not int or MAX_NEW_TOKENS <= 0 or not CAPTION_PROMPT.strip():
    raise ValueError('MAX_NEW_TOKENS must be positive; prompt must not be empty')
if not (math.isfinite(SLOW_MOTION_THRESHOLD) and math.isfinite(TIMELAPSE_THRESHOLD)
        and 0 < SLOW_MOTION_THRESHOLD < TIMELAPSE_THRESHOLD):
    raise ValueError('Need 0 < SLOW_MOTION_THRESHOLD < TIMELAPSE_THRESHOLD')

config_path = BACKEND_DIR / 'configs/caption_extraction.yaml'
cfg = yaml.safe_load(config_path.read_text(encoding='utf-8'))
extraction = cfg['extraction']
extraction.pop('caption_profiles', None)
extraction.pop('internvl', None)
extraction.update(caption_model=CAPTION_MODEL, caption_max_frames=MAX_FRAMES,
                  caption_batch_size=CAPTION_BATCH_SIZE, caption_min_color_bins=MIN_COLOR_BINS, vllm=vllm_config,
                  prompt=CAPTION_PROMPT.strip(), frame_sample_fps=float(FRAME_SAMPLE_FPS),
                  max_new_tokens=MAX_NEW_TOKENS, print_captions=PRINT_CAPTIONS)
extraction['speed_predictor']['timelapse_threshold'] = float(TIMELAPSE_THRESHOLD)
extraction['speed_predictor']['slow_motion_threshold'] = float(SLOW_MOTION_THRESHOLD)
config_path.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
# Runtime batch/VRAM tuning keeps the same output/checkpoint directory.
run_config = {key: value for key, value in extraction.items() if key not in {'caption_batch_size', 'vllm', 'print_captions'}}
run_config['caption_backend'] = 'vllm-internvl-v1'
run_config['frame_filter_version'] = caption_runtime.CAPTION_FRAME_FILTER_VERSION
CAPTION_RUN_ID = hashlib.sha256(yaml.safe_dump(run_config, sort_keys=True).encode()).hexdigest()[:10]
STAGE_DIR = WORK / 'pending_json_2b' / CAPTION_RUN_ID
STAGE_DIR.mkdir(parents=True, exist_ok=True)
print('Caption model:', CAPTION_MODEL, '| backend: vLLM | run:', CAPTION_RUN_ID, flush=True)
print('Sampling FPS:', FRAME_SAMPLE_FPS, '| max frames:', MAX_FRAMES, '| tile: model-native (448)', flush=True)
print('Frame filter: full RGB histogram 8x8x8 before resize, drop bins <', MIN_COLOR_BINS, flush=True)
print('Batch size:', CAPTION_BATCH_SIZE, '| scheduler/VRAM:', vllm_config, flush=True)
print('SpeedPredictor:', extraction['speed_predictor'], flush=True)
if VERBOSE_LOGS:
    print('Caption prompt:', CAPTION_PROMPT, flush=True)
print('Progress: OMNI decode/windows -> SPEED -> CAPTION; warnings/errors always shown.', flush=True)
print('Local captions:', WORK / 'captions_2b' / CAPTION_RUN_ID / DATASET, flush=True)
print('Chạy cell 5 để xử lý; Drive vẫn bỏ qua ID đã có JSON trừ khi OVERWRITE_REMOTE=True.', flush=True)


In [ ]:
# 4. Liệt kê video nguồn và bỏ qua caption JSON đã upload
def remote_args(folder_id):
    return [rclone, '--config', RCLONE_CONFIG, '--drive-root-folder-id', folder_id]

source_names = set(subprocess.check_output(
    [*map(str, remote_args(SOURCE_ROOT_FOLDER_ID)), 'lsf', f'{RCLONE_REMOTE}:', '--files-only'],
    text=True).splitlines())
selected = [video_id(number) for number in range(START_VIDEO_ID, END_VIDEO_ID + 1)]
missing = [vid for vid in selected if f'{vid}{VIDEO_EXTENSION}' not in source_names]
if missing:
    raise FileNotFoundError(f'Thiếu video trong Drive nguồn: {missing}')
dest_names = set(subprocess.check_output(
    [*map(str, remote_args(DEST_ROOT_FOLDER_ID)), 'lsf', f'{RCLONE_REMOTE}:', '--files-only'],
    text=True).splitlines())
pending = [vid for vid in selected if OVERWRITE_REMOTE or f'{vid}.json' not in dest_names]
print(f'Đã chọn {len(selected)} video; cần xử lý {len(pending)}:', pending)


In [ ]:
# 5. Từng lô: tải video cần xử lý, tạo shot + caption, upload JSON rồi dọn video tạm
import math
cfg = yaml.safe_load((BACKEND_DIR / 'configs/caption_extraction.yaml').read_text(encoding='utf-8'))
if cfg['extraction']['caption_model'] != CAPTION_MODEL:
    raise RuntimeError('Caption config trên nhánh này chưa đúng model')
CAPTION_ENV = os.environ.copy()
CAPTION_ENV['PYTHONPATH'] = str(BACKEND_DIR)
CAPTION_ENV['PYTHONUNBUFFERED'] = '1'
CAPTION_ENV['PYTHONFAULTHANDLER'] = '1'
CAPTION_ENV['HF_HOME'] = str(WORK / 'hf_cache')
CAPTION_ENV['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
CAPTION_ENV['TOKENIZERS_PARALLELISM'] = 'false'
CAPTION_ENV['OMP_NUM_THREADS'] = '2'
CAPTION_ENV['CAPTION_VERBOSE'] = '1' if VERBOSE_LOGS else '0'
CAPTION_ENV['CAPTION_PROGRESS_PROTOCOL'] = '1'
CAPTION_ENV['VLLM_LOGGING_LEVEL'] = 'INFO' if VERBOSE_LOGS else 'WARNING'
CAPTION_ENV['HF_HUB_DISABLE_PROGRESS_BARS'] = '0' if VERBOSE_LOGS else '1'
SCENES_DIR = WORK / 'scenes' / DATASET
CAPTIONS_DIR = WORK / 'captions_2b' / CAPTION_RUN_ID / DATASET
SCENES_DIR.mkdir(parents=True, exist_ok=True)
CAPTIONS_DIR.mkdir(parents=True, exist_ok=True)

def validate_json(path, vid):
    data = json.loads(path.read_text(encoding='utf-8'))
    if not isinstance(data, list):
        raise ValueError(f'{path}: cần danh sách caption')
    if not data:
        print(f'{vid}: no caption segments (all sampled frames were filtered); valid empty JSON.', flush=True)
    previous_start = -1.0
    for item in data:
        if (not isinstance(item, dict) or set(item) != {'video_id', 'start', 'end', 'caption'} or
                item['video_id'] != vid or not isinstance(item['caption'], str) or
                not item['caption'].strip() or
                not isinstance(item['start'], (int, float)) or
                not isinstance(item['end'], (int, float)) or
                not math.isfinite(item['start']) or not math.isfinite(item['end']) or
                not 0 <= item['start'] < item['end'] or item['start'] < previous_start):
            raise ValueError(f'{path}: segment không đúng schema')
        previous_start = item['start']
    return len(data)

# Restore staged JSON after an interrupted upload; completed files remain in CAPTIONS_DIR.
for staged in STAGE_DIR.glob('*.json'):
    local = CAPTIONS_DIR / staged.name
    if not local.is_file() or staged.read_bytes() != local.read_bytes():
        raise RuntimeError(f'JSON staged không khớp bản local: {staged}')
    staged.unlink()

# Recheck remote on cell rerun, so completed batches are not uploaded again.
remote_done = set(subprocess.check_output(
    [*map(str, remote_args(DEST_ROOT_FOLDER_ID)), 'lsf', f'{RCLONE_REMOTE}:', '--files-only'],
    text=True).splitlines())
remaining = [vid for vid in selected if OVERWRITE_REMOTE or f'{vid}.json' not in remote_done]
completed = {}
for batch_number, start in enumerate(range(0, len(remaining), UPLOAD_BATCH_SIZE)):
    batch_ids = remaining[start:start + UPLOAD_BATCH_SIZE]
    batch_work = JOBS_ROOT / f'batch_{batch_number:04d}'
    video_dir = batch_work / 'videos' / DATASET
    video_dir.mkdir(parents=True, exist_ok=True)
    need_processing = [vid for vid in batch_ids if not (CAPTIONS_DIR / f'{vid}.json').is_file()]
    for vid in need_processing:
        path = video_dir / f'{vid}{VIDEO_EXTENSION}'
        if not path.is_file() or path.stat().st_size == 0:
            run([*remote_args(SOURCE_ROOT_FOLDER_ID), 'copyto',
                 f'{RCLONE_REMOTE}:{path.name}', path])
        else:
            print('Dùng lại video đã tải:', path.name)
        if not path.is_file() or path.stat().st_size == 0:
            raise RuntimeError(f'Video tải về rỗng: {path.name}')
    if need_processing:
        cmd = [sys.executable, '-u', '-m', 'scripts.caption.extract.run', '--dataset', DATASET.lower(),
               '--input-dir', video_dir, '--scenes-dir', SCENES_DIR, '--output-dir', CAPTIONS_DIR]
        for vid in need_processing:
            cmd += ['--video-id', vid]
        run(cmd, cwd=BACKEND_DIR, env=CAPTION_ENV)
    for vid in batch_ids:
        result = CAPTIONS_DIR / f'{vid}.json'
        if not result.is_file():
            raise FileNotFoundError(f'Không có caption JSON: {result}')
        completed[vid] = validate_json(result, vid)
        shutil.copy2(result, STAGE_DIR / result.name)
        print(f'Caption xong {vid}: {completed[vid]} segment')
    for path in video_dir.glob(f'*{VIDEO_EXTENSION}'):
        path.unlink()  # giữ shot/JSON trong WORK, giải phóng video trước khi upload
    run([*remote_args(DEST_ROOT_FOLDER_ID), 'copy', STAGE_DIR, f'{RCLONE_REMOTE}:',
         '--transfers', str(min(UPLOAD_BATCH_SIZE, len(batch_ids))), '--checkers', '10'])
    run([*remote_args(DEST_ROOT_FOLDER_ID), 'check', STAGE_DIR, f'{RCLONE_REMOTE}:', '--one-way'])
    for path in STAGE_DIR.glob('*.json'):
        path.unlink()
    shutil.rmtree(batch_work)
    print(f'Đã upload {len(batch_ids)} caption JSON; giữ shot/JSON local, dọn video tạm')
print('Hoàn tất:', completed, '| commit:', commit)
